【分析需求】

- 客群貢獻度大盤指標化：跳脫單筆訂單的破碎視角，以「客戶」為核心單元，動態計算 2020 年度所有消費客群的平均年度總消費額，以此作為全大盤的業績基準線。
- 高價值核心門檻篩選：運用WITH(CTE)結合巢狀子查詢的條件過濾流程，精準識別出年度總消費額高於全體平均值的「核心高價值客戶」，硬性排除低頻、低客單的長尾客群。
- 動態消費行為追蹤：在保留原始訂單明細的架構下，結合進階視窗函數 (LAG)，跨越時間序列同步追蹤核心客戶「當前交易金額」與「前次交易金額」的動態差額，將客戶的消費黏性與波動趨勢量化。
- 客戶輪廓多維度關聯：透過JOIN機制整合銷售明細表與客戶基本資料表，補全核心客群的「客戶名稱」與「所屬城市」等維度，為後續的區域精準行銷提供乾淨的數據基礎。

【結果解讀與商業價值】

- 核心高價值客群（VIP）識別：<span style="color: var(--vscode-foreground);">清單中留下的客戶均為 2020 年度貢獻度超越平均線的黃金客群。</span>
- 客戶消費生命週期與波動預警：透過\[差額\]欄位，可以即時監控客戶的訂單行為。若\[差額\]持續為正數，代表該客戶正在「擴大採購規模」，處於黃金成長期；若出現大幅度負數，則為「採購衰退預警」。
- 地理輪廓與區域資源配置：透過\[所屬城市\]的呈現，能清晰看出這些高價值核心客戶主要集中在哪些區域，可協助管理階層評估是否需要在特定核心城市加大物流倉儲、生鮮冷鏈的分撥密度，或調整該地區的業務開發人力配置。
- <span style="color: var(--vscode-foreground);">商業價值：</span>

1. <span style="color: var(--vscode-foreground);">企業應優先將 80% 的客戶關係維護資源（如專屬客戶經理、客製化讓利、優先配貨權）配置於此名單，確保核心營收大盤的穩定。</span>
2. <span style="color: var(--vscode-foreground);">當<span data-path-to-node="16,1,0" data-index-in-node="7" style="">此分析報表定期產出並呈現</span>核心客戶的[差額]連續兩筆為負且金額擴大時，業務團隊可在第一時間介入關懷，防範核心客戶流失。</span>

In [31]:
--年度每筆訂單與前客單價對比
WITH CTE_1 AS (SELECT id,
                   employee_id,
                   client_id,
                   product_id,
                   Amount,
                   [Date],
                   LAG(Amount) OVER(partition by client_id order by Date) AS [上一筆訂單金額],
                   Amount-LAG(Amount) OVER(partition by client_id order by Date) AS [差額]
              FROM HappyFruit.dbo.Sales
             WHERE [Date] like '2020年%'),

--客戶年度總消費
CTE_2 AS (SELECT client_id,
                 SUM(Amount) AS [年度總消費額]
            FROM HappyFruit.dbo.Sales
           WHERE [Date] like '2020年%'
           GROUP BY client_id)
--主查詢
SELECT C1.id AS [訂單編號],
       C1.Date AS [消費日期],
       Cli.Name AS [客戶名稱],
       Cli.city AS [所屬城市],
       C1.Amount AS [當次消費金額],
       C1.[上一筆訂單金額],
       C1.[差額]
  FROM cte_1 AS C1
  JOIN HappyFruit.dbo.Clients AS Cli
    ON C1.client_id=cli.id
 WHERE C1.client_id in (SELECT Client_id
                          FROM CTE_2
                         WHERE [年度總消費額]>(SELECT AVG([年度總消費額])
                                                FROM CTE_2)
)
 ORDER BY Cli.Name,C1.Date;

(251 rows affected)

Total execution time: 00:00:00.091

訂單編號,消費日期,客戶名稱,所屬城市,當次消費金額,上一筆訂單金額,差額
607,2020年10月14日,69小華超商,臺北市,76000.00,NULL,NULL
609,2020年10月16日,69小華超商,臺北市,225000.00,76000.00,149000.00
614,2020年10月21日,69小華超商,臺北市,202000.00,225000.00,-23000.00
600,2020年10月5日,69小華超商,臺北市,163000.00,202000.00,-39000.00
602,2020年10月6日,69小華超商,臺北市,251000.00,163000.00,88000.00
636,2020年11月17日,69小華超商,臺北市,157000.00,251000.00,-94000.00
624,2020年11月1日,69小華超商,臺北市,262000.00,157000.00,105000.00
640,2020年11月22日,69小華超商,臺北市,241000.00,262000.00,-21000.00
656,2020年12月15日,69小華超商,臺北市,68000.00,241000.00,-173000.00
658,2020年12月19日,69小華超商,臺北市,173000.00,68000.00,105000.00
